# Part D: Data Preparation

### D0. Load and preprocess

In [2]:
import torch
import torch.nn as nn
import nltk
import re

#### Load sentences

In [3]:
from nltk.corpus import brown

brown_sentences = brown.sents(categories='news')
brown_sentences

[['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.'], ['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.'], ...]

#### Preprocess tokens

In [4]:
browns = []
for sentences in brown_sentences:
    # Lowecase and remove punctuation-only tokens
    browns.append([sent.lower() for sent in sentences if re.search(r'[a-zA-Z]', sent)])

    
brown_sentences = browns

# keeping stopwords which is required for language modeling
brown_sentences

[['the',
  'fulton',
  'county',
  'grand',
  'jury',
  'said',
  'friday',
  'an',
  'investigation',
  'of',
  "atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  'no',
  'evidence',
  'that',
  'any',
  'irregularities',
  'took',
  'place'],
 ['the',
  'jury',
  'further',
  'said',
  'in',
  'term-end',
  'presentments',
  'that',
  'the',
  'city',
  'executive',
  'committee',
  'which',
  'had',
  'over-all',
  'charge',
  'of',
  'the',
  'election',
  'deserves',
  'the',
  'praise',
  'and',
  'thanks',
  'of',
  'the',
  'city',
  'of',
  'atlanta',
  'for',
  'the',
  'manner',
  'in',
  'which',
  'the',
  'election',
  'was',
  'conducted'],
 ['the',
  'september-october',
  'term',
  'jury',
  'had',
  'been',
  'charged',
  'by',
  'fulton',
  'superior',
  'court',
  'judge',
  'durwood',
  'pye',
  'to',
  'investigate',
  'reports',
  'of',
  'possible',
  'irregularities',
  'in',
  'the',
  'hard-fought',
  'primary',
  'which',
  'was',
  'won',
 

#### adding special tokens like `<bos>, <eos> and <unk>`

In [5]:
browns_tokens = []
for sent in brown_sentences:
    if len(sent)>0:
        sent = ['<bos>']+sent+['<eos>']
    browns_tokens.append(sent)
brown_sentences = browns_tokens
brown_sentences

[['<bos>',
  'the',
  'fulton',
  'county',
  'grand',
  'jury',
  'said',
  'friday',
  'an',
  'investigation',
  'of',
  "atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  'no',
  'evidence',
  'that',
  'any',
  'irregularities',
  'took',
  'place',
  '<eos>'],
 ['<bos>',
  'the',
  'jury',
  'further',
  'said',
  'in',
  'term-end',
  'presentments',
  'that',
  'the',
  'city',
  'executive',
  'committee',
  'which',
  'had',
  'over-all',
  'charge',
  'of',
  'the',
  'election',
  'deserves',
  'the',
  'praise',
  'and',
  'thanks',
  'of',
  'the',
  'city',
  'of',
  'atlanta',
  'for',
  'the',
  'manner',
  'in',
  'which',
  'the',
  'election',
  'was',
  'conducted',
  '<eos>'],
 ['<bos>',
  'the',
  'september-october',
  'term',
  'jury',
  'had',
  'been',
  'charged',
  'by',
  'fulton',
  'superior',
  'court',
  'judge',
  'durwood',
  'pye',
  'to',
  'investigate',
  'reports',
  'of',
  'possible',
  'irregularities',
  'in',
  'the',
  'ha

In [6]:
from collections import Counter

word_counts = Counter()

for sent in brown_sentences:
    word_counts.update(sent)

# Minimum frequency threshold (optional)
min_freq = 2

vocab = {
    word for word, freq in word_counts.items()
    if freq >= min_freq
}

# Ensure special tokens are included
vocab.update(['<bos>', '<eos>', '<unk>'])

print("Vocab size:", len(vocab))
vocab

Vocab size: 6080


{'distinction',
 'extension',
 'puerto',
 'meals',
 'write',
 'badly',
 'conference',
 'indication',
 'original',
 "today's",
 'generation',
 'eradication',
 'lawn',
 'look',
 'sanctuary',
 'simply',
 'birdied',
 'skorich',
 'sending',
 'devoted',
 'rotary',
 'consequently',
 'supplied',
 'hard',
 'matters',
 'formation',
 'three',
 'greeting',
 'closer',
 'issue',
 'products',
 'uncertain',
 'gladden',
 'coins',
 'superior',
 "o'sullivan",
 'lighter',
 'beauty',
 'monte',
 'communities',
 'elaborate',
 'pockets',
 'ideally',
 'keegan',
 'revision',
 'rising',
 'supper',
 'awareness',
 'killed',
 'remember',
 'active',
 'colony',
 'farms',
 'proposal',
 '13th',
 'sat',
 'nikita',
 'acceptable',
 'pro-communist',
 'world',
 'principal',
 'states',
 'spectator',
 'mantle',
 '19th',
 'voiced',
 'politely',
 'thirteen',
 'humor',
 'thailand',
 'shopping',
 'printed',
 'vernava',
 'title',
 'footnotes',
 'hammered',
 'is',
 'blow',
 'meyner',
 'geraghty',
 'hardwicke-etter',
 'orderly',
 's

In [7]:
final_sentences = []

for sent in brown_sentences:
    new_sent = [
        word if word in vocab else '<unk>'
        for word in sent
    ]
    final_sentences.append(new_sent)
final_sentences

[['<bos>',
  'the',
  'fulton',
  'county',
  'grand',
  'jury',
  'said',
  'friday',
  'an',
  'investigation',
  'of',
  "atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  'no',
  'evidence',
  'that',
  'any',
  'irregularities',
  'took',
  'place',
  '<eos>'],
 ['<bos>',
  'the',
  'jury',
  'further',
  'said',
  'in',
  '<unk>',
  '<unk>',
  'that',
  'the',
  'city',
  'executive',
  'committee',
  'which',
  'had',
  'over-all',
  'charge',
  'of',
  'the',
  'election',
  'deserves',
  'the',
  'praise',
  'and',
  'thanks',
  'of',
  'the',
  'city',
  'of',
  'atlanta',
  'for',
  'the',
  'manner',
  'in',
  'which',
  'the',
  'election',
  'was',
  'conducted',
  '<eos>'],
 ['<bos>',
  'the',
  '<unk>',
  'term',
  'jury',
  'had',
  'been',
  'charged',
  'by',
  'fulton',
  'superior',
  'court',
  'judge',
  '<unk>',
  '<unk>',
  'to',
  'investigate',
  'reports',
  'of',
  'possible',
  'irregularities',
  'in',
  'the',
  '<unk>',
  'primary',
  '

#### D1. Split

In [8]:
import random

random.seed(42)
random.shuffle(final_sentences)

In [9]:
total = len(final_sentences)

train_size = int(0.8 * total)
val_size   = int(0.1 * total)

train_sentences = final_sentences[:train_size]
val_sentences   = final_sentences[train_size:train_size + val_size]
test_sentences  = final_sentences[train_size + val_size:]

print("Sentence counts:")
print("Train:", len(train_sentences))
print("Validation:", len(val_sentences))
print("Test:", len(test_sentences))

Sentence counts:
Train: 3698
Validation: 462
Test: 463


In [10]:
def count_tokens(sentences):
    return sum(len(sent) for sent in sentences)

print("\nToken counts:")
print("Train tokens:", count_tokens(train_sentences))
print("Validation tokens:", count_tokens(val_sentences))
print("Test tokens:", count_tokens(test_sentences))


Token counts:
Train tokens: 77203
Validation tokens: 9396
Test tokens: 9617


#### D2. Shared Vocabulary (Train Only)

In [11]:
min_freq = 2
from collections import Counter

word_counts = Counter()

for sent in train_sentences:
    word_counts.update(sent)

In [12]:
vocab = {
    word for word, freq in word_counts.items()
    if freq >= min_freq
}

# Ensure special tokens are present
vocab.update(['<bos>', '<eos>', '<unk>'])

print("Chosen min_freq:", min_freq)
print("Final vocabulary size:", len(vocab))

Chosen min_freq: 2
Final vocabulary size: 5270


In [13]:
def replace_unk(sentences, vocab):
    return [
        [word if word in vocab else '<unk>' for word in sent]
        for sent in sentences
    ]

train_sentences = replace_unk(train_sentences, vocab)
val_sentences   = replace_unk(val_sentences, vocab)
test_sentences  = replace_unk(test_sentences, vocab)

In [14]:
top_20 = word_counts.most_common(20)

print("\nTop 20 most frequent tokens:")
for word, freq in top_20:
    print(word, freq)


Top 20 most frequent tokens:
<unk> 5163
the 5116
<bos> 3685
<eos> 3685
of 2287
and 1768
a 1720
to 1716
in 1623
for 786
that 658
is 582
was 574
on 535
he 514
at 513
with 459
as 427
be 422
by 409


## Part A – Statistical n-gram Language Model 

#### A1. Trigram language model

In [15]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE
from nltk.lm import Laplace
from nltk.lm import Lidstone

n = 3  # trigram model

train_data, padded_vocab = padded_everygram_pipeline(n, train_sentences)

In [16]:
from nltk.lm.preprocessing import padded_everygram_pipeline

def compute_perplexity(model, sentences, n=3):
    test_data, _ = padded_everygram_pipeline(n, sentences)

    # Flatten generator of generators
    test_ngrams = [ngram for sent in test_data for ngram in sent]

    return model.perplexity(test_ngrams)



In [17]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Lidstone

n = 3

for gamma in [1.0, 0.1, 0.01]:
    train_data, padded_vocab = padded_everygram_pipeline(n, train_sentences)
    
    model = Lidstone(gamma, n)
    model.fit(train_data, padded_vocab)
    
    val_ppl = compute_perplexity(model, val_sentences, n)
    
    print(f"Gamma={gamma}, Validation Perplexity={val_ppl}")

Gamma=1.0, Validation Perplexity=517.1259356972927
Gamma=0.1, Validation Perplexity=356.96040280037596
Gamma=0.01, Validation Perplexity=305.20226773438515


In [18]:
print("Train Perplexity:", compute_perplexity(model, train_sentences))
print("Validation Perplexity:", compute_perplexity(model, val_sentences))
print("Test Perplexity:", compute_perplexity(model, test_sentences))

Train Perplexity: 80.07081743110142
Validation Perplexity: 305.20226773438515
Test Perplexity: 307.6386788402556


Report:
- Which model you usedtrained a Trigram Language Model using NLTK’s nltk.lm module.
i trained a Trigram Language Model using NLTK’s nltk.lm module.

Which smoothing method (if any)
I used Lidstone because it is more smoother where we can expreiment with different gamma. Here gamma = 0.01 is the best.


Why smoothing is necessary
in our case smooting is essential because there were cases of trigrams getting probability 0. which meant that perplexity in validation test becomes infinity. To eliminate this problem smoothing is used because any unseen words are distributed with small probabilities which produces stable perplexities.

Report:
1. Validation Perplexity: 305.20226773438515

Test Perplexity: 307.6386788402556

2. Based on our dataset this seems to be good, for dataset which has vocab of 5200, perplexity of 305.2 is good enough.

3. some of the factors are :
- dataset size
- model order (n - gram)
- smoothing strength ( gamma )

In [20]:
import random

def generate_sentence(model, max_tokens=50):
    context = ["<bos>", "<bos>"]
    generated = []

    for _ in range(max_tokens):
        next_word = model.generate(text_seed=context[-2:])
        
        if next_word == "<eos>":
            break
            
        generated.append(next_word)
        context.append(next_word)

    return " ".join(generated)

In [21]:
for i in range(3):
    text = generate_sentence(model, max_tokens=60)
    
    # Ensure minimum length
    while len(text.split()) < 30:
        text = generate_sentence(model, max_tokens=60)
    
    print(f"\nSample {i+1}:\n{text}\n")


Sample 1:
in nixon delivered a peppery annual report in the comedy division the kings george worth bill kay frank <unk> have had integrated student bodies but their tax-exempt status never has been agreed upon


Sample 2:
there may be short of their admission to said university and would only have to toss in the atlanta negro student movement renewed its demands for movie theater integration several months ago


Sample 3:
russian tanks and artillery <unk> fidel castro who had a stroke yesterday while in the cold <unk> three members of the ogden standard examiner went to press russia for a definite contemporary <unk> keynote <unk> fall group composite



Report:
1. Are they grammatical?

    They are grammatical 

2. Are they cohorent?

    They are not cohorent

3. What limitations do you observe?

    That in real world this is not useful because how incohorent it is. Also theres topic drift. 